# gammaConversionSi — training the conversion DNN

Trains `ConversionDNN` from `conversion_dnn.py` on the conversion ntuple.

    inputs   eGamma, pathInBlock
    outputs  eLead, thetaLead, eRecoil
    derived  eSub = eGamma - 2*me - eLead - eRecoil

The pair is sorted by energy rather than by charge, so `eLead`/`eSub` are the higher/lower
lepton energy and `thetaLead` is the angle of whichever lepton carried `eLead`.

`eRecoil` is predicted rather than neglected, which is what makes the energy-conservation step
exact. The recoil is the nucleus for ~93 % of conversions and carries only tens of eV, but for
the remaining ~1/(Z+1) it is an atomic electron carrying up to two thirds of the photon energy.

**Normalisation is inside the model**, as non-trained buffers fitted from the training split:
`log10` followed by a min-max rescaling onto `[0, 1]`.
There is no scaler in this notebook, and a saved `state_dict` is self-contained.

Nothing is drawn inline — every figure is saved and closed.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import torch
import uproot
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from conversion_dnn import ConversionDNN, build_dataset, ELECTRON_MASS

hep.style.use(hep.style.ATLAS)

# TeX Gyre Heros has no sized-radical glyph, so the \sqrt in the ATLAS `com`
# label warns and falls back to '?'. stixsans supplies it; fontset stays
# "custom", so all other math is still TeX Gyre Heros.
plt.rcParams["mathtext.fallback"] = "stixsans"


In [ ]:
def find_repo_root(start=None):
    """Walk up from `start` until a directory containing .git is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"no .git found above {path}")


REPO = find_repo_root()

NTUPLE = REPO / "build/gammaConversionSi/ntuples/Si_2MeV-100GeV_10000000.root"
PLOTS = REPO / "studies/gammaConversionSi/training/plots"
MODELS = REPO / "studies/gammaConversionSi/training/models"

# Feature plots go in their own subdirectory of plots/, named here
INPUT_PLOT_SUBDIR = "training_inputs"
INPUT_PLOTS = PLOTS / INPUT_PLOT_SUBDIR

COM = "2MeV - 100GeV"

# Training configuration
MAX_EVENTS = None  # set e.g. 500_000 to iterate quickly
VAL_FRACTION = 0.2
BATCH_SIZE = 8192
EPOCHS = 20
LEARNING_RATE = 1e-3
SEED = 0

for directory in (INPUT_PLOTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

if not NTUPLE.exists():
    raise FileNotFoundError(
        f"{NTUPLE} not found. Generate it from the repository root with:\n"
        f"    ./studies/gammaConversionSi/build.sh\n"
        f"    source install/geant4/bin/geant4.sh\n"
        f"    cd build/gammaConversionSi && ./gammaConversionSi config/default.cfg"
    )

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

print(f"repository  : {REPO}")
print(f"ntuple      : {NTUPLE.relative_to(REPO)}")
print(f"feature plots: {INPUT_PLOTS.relative_to(REPO)}")
print(f"device      : {DEVICE}")

In [ ]:
tree = uproot.open(NTUPLE)["conversions"]
arrays = tree.arrays(
    ["eGamma", "pathInBlock", "eElectron", "ePositron",
     "thetaElectron", "thetaPositron", "eRecoil"],
    entry_stop=MAX_EVENTS,
    library="np",
)

x_all, y_all, extra = build_dataset(arrays)
e_sub_true = extra["eSub"]

print(f"events: {len(x_all):,}")
print(f"inputs  {x_all.shape}  targets {y_all.shape}")

In [ ]:
def plot_feature(values, xlabel, filename, logy=False, logx=True, bins=100):
    """Histogram one training variable and save it into INPUT_PLOTS.

    Same style as the exploration notebook: black outline, no fill, Poisson
    error bars, saved and closed rather than shown.
    """
    values = np.asarray(values, dtype=float)

    if logx:
        values = values[values > 0]
        edges = np.logspace(np.log10(values.min()), np.log10(values.max()), bins + 1)
    else:
        edges = np.linspace(values.min(), values.max(), bins + 1)

    counts, edges = np.histogram(values, bins=edges)

    fig, ax = plt.subplots()
    hep.histplot(counts, edges, ax=ax, yerr=True, histtype="step", color="black")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Conversions")
    if logx:
        ax.set_xscale("log")

    peak = counts.max() + np.sqrt(counts.max())
    if logy:
        ax.set_yscale("log")
        floor = max(counts[counts > 0].min(), 1) if (counts > 0).any() else 1
        ax.set_ylim(0.5 * floor, peak * 10)
    else:
        ax.set_ylim(0, peak * 1.35)

    hep.atlas.label(text="Internal", com=COM, ax=ax)

    path = INPUT_PLOTS / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


FEATURES = [
    # model inputs
    (x_all[:, 0], r"$E_\gamma$ [MeV]",                 "input_eGamma.pdf",      dict()),
    (x_all[:, 1], r"Path in block [mm]",               "input_pathInBlock.pdf", dict(logy=True)),
    # model targets
    (y_all[:, 0], r"$E_\mathrm{lead}$ [MeV]",          "target_eLead.pdf",      dict()),
    (y_all[:, 1], r"$\theta_\mathrm{lead}$ [rad]",     "target_thetaLead.pdf",  dict(logy=True)),
    (y_all[:, 2], r"$E_\mathrm{recoil}$ [MeV]",        "target_eRecoil.pdf",    dict(logy=True)),
    # derived, for reference
    (e_sub_true,  r"$E_\mathrm{sub}$ [MeV]",           "derived_eSub.pdf",      dict(logy=True)),
]

for values, xlabel, filename, kwargs in FEATURES:
    path = plot_feature(values, xlabel, filename, **kwargs)
    print(f"saved {path.relative_to(REPO)}")

In [ ]:
# Shuffle once, then split. The normalisation is fitted on the training half
# only -- fitting on everything would leak the validation set's range.
order = rng.permutation(len(x_all))
n_val = int(len(order) * VAL_FRACTION)
val_idx, train_idx = order[:n_val], order[n_val:]

x_train = torch.from_numpy(x_all[train_idx])
y_train = torch.from_numpy(y_all[train_idx])
x_val = torch.from_numpy(x_all[val_idx]).to(DEVICE)
y_val = torch.from_numpy(y_all[val_idx]).to(DEVICE)

model = ConversionDNN().to(DEVICE)
model.fit_normalisation(x_train.to(DEVICE), y_train.to(DEVICE))

print(f"train {len(x_train):,}   validation {len(x_val):,}")
# Everything is mapped onto [0, 1] from these log10 ranges. Validation rows
# outside the training range decode to slightly outside [0, 1], by design.
print(f"input  log10 min {model.input_min.tolist()}")
print(f"input  log10 max {model.input_max.tolist()}")
print(f"target log10 min {model.target_min.tolist()}")
print(f"target log10 max {model.target_max.tolist()}")

In [ ]:
loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,   # BatchNorm1d needs more than one sample per batch
)

optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()


@torch.no_grad()
def evaluate(x, y, chunk=131072):
    """Validation loss, in chunks.

    The validation split is ~2M rows; pushing it through in one batch would
    allocate gigabytes of hidden activations at once.
    """
    model.eval()
    total, count = 0.0, 0
    for start in range(0, len(x), chunk):
        xb, yb = x[start:start + chunk], y[start:start + chunk]
        total += loss_fn(model.forward_normalised(xb), model.normalise_targets(yb)).item() * len(xb)
        count += len(xb)
    return total / count


history = {"train": [], "val": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, batches = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        # Loss in normalised space: the three targets span 5.5, 7.5 and 12
        # orders of magnitude, so a physical-units loss would be all eRecoil
        # tail. Rescaled onto [0, 1] they weigh equally
        loss = loss_fn(model.forward_normalised(xb), model.normalise_targets(yb))

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running += loss.item()
        batches += 1

    val_loss = evaluate(x_val, y_val)

    history["train"].append(running / batches)
    history["val"].append(val_loss)
    print(f"epoch {epoch:3d}   train {history['train'][-1]:.5f}   val {val_loss:.5f}")

In [ ]:
fig, ax = plt.subplots()
epochs = np.arange(1, len(history["train"]) + 1)
ax.plot(epochs, history["train"], color="black", label="Train")
ax.plot(epochs, history["val"], color="black", linestyle="--", label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE ([0, 1] normalised space)")
ax.set_yscale("log")
ax.legend()
hep.atlas.label(text="Internal", com=COM, ax=ax)
fig.savefig(PLOTS / "training_loss.pdf", bbox_inches="tight")
plt.close(fig)

print(f"first epoch val {history['val'][0]:.5f}  ->  last {history['val'][-1]:.5f}")
print(f"saved {(PLOTS / 'training_loss.pdf').relative_to(REPO)}")

In [ ]:
@torch.no_grad()
def predict(x, chunk=131072):
    """Physical predictions for a large input tensor, in chunks."""
    model.eval()
    parts = [model(x[start:start + chunk]) for start in range(0, len(x), chunk)]
    return {key: torch.cat([p[key] for p in parts]) for key in parts[0]}


pred = predict(x_val)

truth = {"eLead": y_val[:, 0], "thetaLead": y_val[:, 1], "eRecoil": y_val[:, 2]}

print("median fractional error on the validation set")
for name, true in truth.items():
    frac = ((pred[name] - true).abs() / true.clamp(min=1e-30)).median().item()
    print(f"  {name:10} {frac:.3f}")

# eSub is derived, not predicted, so nothing forces it positive. The fraction
# that comes out negative is a blunt convergence check.
negative = (pred["eSub"] < 0).float().mean().item()
print(f"\nvalidation events with eSub < 0: {negative * 100:.2f} %")

# Conservation is exact by construction, whatever the network predicts
residual = (
    x_val[:, 0] - 2 * ELECTRON_MASS - pred["eLead"] - pred["eRecoil"] - pred["eSub"]
).abs().max().item()
print(f"energy conservation residual: {residual:.3e} MeV")

In [ ]:
path = MODELS / "conversion_dnn.pt"
torch.save(model.state_dict(), path)

# The state_dict carries the normalisation buffers, so this is all that is
# needed for inference -- no scaler to reload alongside it.
reloaded = ConversionDNN().to(DEVICE)
reloaded.load_state_dict(torch.load(path))
reloaded.eval()
with torch.no_grad():
    check = reloaded(x_val[:1024])

identical = all(torch.equal(check[k], pred[k][:1024]) for k in check)
print(f"saved {path.relative_to(REPO)}")
print(f"reloaded model reproduces predictions exactly: {identical}")